## Inputs


## A joint payoff table is an explicit finite model

The visible native root loads two asset registers, composes a joint payoff
rotation and the requested Grover iterations. FlagQuantum constructs the
shared reversible preparation/reflection helpers. Host code regenerates the
rounded coefficients, selectors and payoff angles; it does not ask synth to
infer the financial formula or an estimation strategy.

Try a different admitted power and compare the independent full state before
interpreting indicator mass. Preparation and estimation retain width ceilings
14 and 25. The count-only iterative estimator rescales probability and its
interval into price; the eight-entry payoff grid introduces separate model
error. Dropped assets remain private storage. Agreement within the declared
source tolerance is not a real-market valuation guarantee.


In [ ]:
from flagquantum import synth
from flagquantum.synth import Output
from flagquantum.synth import QArray
from flagquantum.synth import QBit
from flagquantum.synth import allocate
from flagquantum.synth import drop
from flagquantum.synth import phase
from flagquantum.synth import qfunc

from examples.synthesis.applications.finance.rainbow_options.rainbow_options_bruteforce_method.main import (
    pi,
    prepare,
    reflect,
)
from examples.synthesis.applications.finance.rainbow_options.rainbow_options_bruteforce_method.overview.workflow import (
    Model,
)


@qfunc
def rainbow(
    k: int, probabilities: list[float], angles: list[float], indicator: Output[QBit]
) -> None:
    asset: QArray[QBit] = QArray()
    allocate(4, asset)
    allocate(indicator)
    prepare(probabilities, angles, asset, indicator)
    for i in range(k):
        phase(indicator, pi)
        reflect(probabilities, angles, asset, indicator)
    drop(asset)


def build_preview(model: Model, power: int, max_width: int) -> synth.Program:
    """Bind the freshly generated tables within the original width budgets."""
    if type(power) is not int or not 0 <= power <= 16:
        raise ValueError("Use a bounded Grover power in 0..16")
    if max_width not in (14, 25):
        raise ValueError("Use the original preparation or estimation width budget")
    return synth.synthesize(
        rainbow,
        classical_inputs=dict(
            k=power, probabilities=model.probabilities, angles=model.angles
        ),
        max_qubits=max_width,
        max_workspace=0,
        max_operations=8192,
        memory_limit_bytes=1024**2,
    )


from IPython.display import Image, display

from examples.synthesis.applications.finance.rainbow_options.rainbow_options_bruteforce_method.overview.main import (
    report,
    save_figure,
)
from examples.synthesis.applications.finance.rainbow_options.rainbow_options_bruteforce_method.overview.model import (
    ALPHA,
    ASSERTION_ALPHA,
    EPSILON,
    SOURCE_PRICE,
    STRIKE,
    fresh_model,
)
from examples.synthesis.applications.finance.rainbow_options.rainbow_options_bruteforce_method.overview.workflow import (
    build_program,
    run_pricing,
)


## Correlated log-return model


In [ ]:
model = fresh_model()
print("Rounded coefficients:", model.coefficients)


## Original estimation settings


In [ ]:
print("Strike, epsilon and alpha:", STRIKE, EPSILON, ALPHA)


## Discretized Gaussian bins


In [ ]:
print("Grid and normalized probabilities:", model.grid, model.probabilities)


## Examine the payoff support


In [ ]:
print("Payoff grid:", model.payoff_grid)


## Fractional precision


In [ ]:
print("Rounded offset:", model.offset)


## Construct the affine maximum


In [ ]:
print("All joint selectors:", model.selectors)


## Three-bit selector bounds


In [ ]:
assert len(model.payoff_grid) == 8 and all(0 <= index < 8 for index in model.selectors)


## Payoff normalization


In [ ]:
print("Payoff scale and joint RY angles:", model.scale, model.angles)


## Synthesize the initial preparation


In [ ]:
preparation = build_preview(model, 0, 14)
print("Preparation resources:", preparation.report["resources"])


## Estimator circuit bounds


In [ ]:
print("Estimation resources:", build_program(model, 0, 25).report["resources"])


## Execute the count-only estimator


In [ ]:
result = run_pricing()
report(result)


## Independent discrete expectation


In [ ]:
print("Exact finite expectation:", model.exact_price)


## Inspect the original model and payoff


In [ ]:
display(Image(filename=str(save_figure(result))))


## Source confidence rule


In [ ]:
assert abs(result.price - SOURCE_PRICE) <= result.source_tolerance
print("Reference confidence level:", 1 - ASSERTION_ALPHA)
